In [5]:
import pandas as pd
# 定义通用大文件读取函数
def read_big_csv(file_path, chunk_size=10000):
    chunk_list = []
    # 分块读取+容错编码
    for chunk in pd.read_csv(file_path, chunksize=chunk_size,
                             encoding="utf-8-sig", encoding_errors="ignore"):
        chunk_list.append(chunk)
    df = pd.concat(chunk_list, ignore_index=True)
    return df
df = pd.read_csv("./data/feature/full_feature_data.csv", encoding="utf-8-sig")

In [6]:
# 统计商品核心指标
item_data = df.groupby("product_id").agg(
    浏览量=("user_id", "count"),
    销量=("quantity", "sum"),
    销售额=("total_amount", "sum")
).reset_index()

# 行业标准权重计算综合热度
item_data["综合热度"] = item_data["浏览量"]*0.2 + item_data["销量"]*0.5 + item_data["销售额"]*0.3
# 热度排序
item_hot_rank = item_data.sort_values("综合热度", ascending=False)
print("===== 商品综合热度TOP10 =====")
print(item_hot_rank.head(10))

===== 商品综合热度TOP10 =====
      product_id  浏览量  销量  销售额  综合热度
1687        9248    3   7    0   4.1
1766        9604    2   7    0   3.9
1709        9313    3   6    0   3.6
331         1750    3   6    0   3.6
215         1184    2   6    0   3.4
1191        6309    2   6    0   3.4
378         2001    2   6    0   3.4
1033        5471    2   6    0   3.4
1379        7433    2   6    0   3.4
1609        8753    2   6    0   3.4


In [7]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# 构建特征与标签
X = df[["order_hour", "price"]]
y = df["quantity"]

# 模型训练
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X, y)

# 预测与评估
y_pred = model.predict(X)
print(f"模型拟合优度R²：{r2_score(y, y_pred):.2f}")
print(f"平均绝对误差MAE：{mean_absolute_error(y, y_pred):.2f}")

模型拟合优度R²：0.69
平均绝对误差MAE：0.29


In [8]:
print(df.head())

   user_id  product_id  quantity  total_amount  status_x         created_at_x  \
0     3094        7999         1             0         2  2026-04-02 03:05:08   
1    18933        2621         1             0         3  2025-08-21 23:25:46   
2     9649        2413         2             0         3  2025-12-05 21:44:49   
3    22563        5268         1             0         3  2025-01-20 14:58:10   
4     3311        6495         5             0         3  2026-03-07 01:45:21   

   order_hour  order_month           name  category_id  ... nickname  gender  \
0           3            4          格力 电视         32.0  ...      龙丽娟       1   
1          23            8         顾家 四件套         73.0  ...      陈凤兰       2   
2          21           12          苹果 U盘         22.0  ...       李云       2   
3          14            1  湾仔码头 咖啡 2025款         63.0  ...       李丽       2   
4           1            3         小米 微波炉         32.0  ...       詹琴       2   

   age  city        register_tim